# lsu-pria — Multimodal Training en Google Colab

Notebook orientado al baseline **multimodal temporal** para LSU:
- mano izquierda,
- mano derecha,
- pose superior,
- cara reducida.

Cubre dos modos:
- **own_multimodal**: secuencias propias ya recolectadas,
- **ilsut_multimodal**: preparación + entrenamiento desde iLSU-T extraído en Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/SEU_USUARIO_O_FORK/lsu-pria.git'
REPO_BRANCH = 'main'
WORKDIR = '/content/lsu-pria'

DATASET_MODE = 'own_multimodal'  # own_multimodal | ilsut_multimodal
DRIVE_OUT_ROOT = '/content/drive/MyDrive/lsu_pria_runs'

# === Own multimodal ===
OWN_MM_SEQ_DIR = '/content/drive/MyDrive/lsu_pria_data/mm_seq_S1'
OWN_MM_SEQ_DIRS = []
OWN_MM_WORKDIR = f'{DRIVE_OUT_ROOT}/train_mm_stack_colab'
OWN_MM_GROUP_COL = 'subject_id'

# === iLSU-T multimodal ===
ILSUT_ROOT = '/content/drive/MyDrive/iLSUT_extracted'
ILSUT_SOURCES = ['source2']
ILSUT_KEYWORDS_JSON = '/content/drive/MyDrive/lsu_pria_data/ilsut_keywords.json'
ILSUT_MM_WORKDIR = f'{DRIVE_OUT_ROOT}/ilsut_mm_colab'
ILSUT_FPS = 4
ILSUT_MAX_PER_SEG = 24
ILSUT_WINDOW = 24
ILSUT_MIN_FRAMES = 8
PREPROCESS = True


In [ ]:
from pathlib import Path

workdir = Path(WORKDIR)
if not workdir.exists():
    !git clone -b {REPO_BRANCH} {REPO_URL} {WORKDIR}

%cd {WORKDIR}
!python3 -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .
!nvidia-smi || true


In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Modo A — dataset propio multimodal

In [ ]:
import subprocess
from pathlib import Path

if DATASET_MODE == 'own_multimodal':
    Path(OWN_MM_WORKDIR).mkdir(parents=True, exist_ok=True)
    cmd = [
        'python3', 'lsupria.py', 'train-mm-stack',
        '--work-dir', OWN_MM_WORKDIR,
        '--group-col', OWN_MM_GROUP_COL,
    ]
    if OWN_MM_SEQ_DIRS:
        cmd += ['--seq-dirs', *OWN_MM_SEQ_DIRS]
    else:
        cmd += ['--seq-dir', OWN_MM_SEQ_DIR]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping own_multimodal')


## Modo B — iLSU-T multimodal

In [ ]:
import subprocess
from pathlib import Path

if DATASET_MODE == 'ilsut_multimodal':
    Path(ILSUT_MM_WORKDIR).mkdir(parents=True, exist_ok=True)
    cmd = [
        'python3', 'lsupria.py', 'ilsut-train-mm',
        '--root', ILSUT_ROOT,
        '--keywords', ILSUT_KEYWORDS_JSON,
        '--work-dir', ILSUT_MM_WORKDIR,
        '--fps', str(ILSUT_FPS),
        '--max-per-seg', str(ILSUT_MAX_PER_SEG),
        '--window', str(ILSUT_WINDOW),
        '--min-frames', str(ILSUT_MIN_FRAMES),
        '--sources', *ILSUT_SOURCES,
    ]
    if PREPROCESS:
        cmd.append('--preprocess')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping ilsut_multimodal')


In [ ]:
from pathlib import Path
import json

work_dir = Path(OWN_MM_WORKDIR if DATASET_MODE == 'own_multimodal' else ILSUT_MM_WORKDIR)
print('work_dir =', work_dir)
for p in sorted(work_dir.rglob('*')):
    if p.is_file():
        print(p)

stats_json = work_dir / 'results' / 'multimodal_dataset_stats.json'
if stats_json.exists():
    print('\n===== multimodal_dataset_stats.json =====\n')
    print(json.dumps(json.loads(stats_json.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))


In [ ]:
import shutil
from pathlib import Path

work_dir = Path(OWN_MM_WORKDIR if DATASET_MODE == 'own_multimodal' else ILSUT_MM_WORKDIR)
zip_path = shutil.make_archive(str(work_dir), 'zip', root_dir=str(work_dir))
print('ZIP generado:', zip_path)
